In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

# 1. Cargar el subset limpio en inglés
df = pd.read_csv('../data/processed/reviews_en_clean.csv')

# 2. Definir X (Texto) y y (Target convertido a entero 0 o 1)
X = df['review']
y = df['recommended'].astype(int)

# 3. Train / Test Split (80/20) con Estratificación, se aplica por la clase objetivo para mantener la proporción de clases en ambos conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.20, 
    random_state=42, 
    stratify=y
)

print(f"Reseñas de Entrenamiento: {len(X_train)}")
print(f"Reseñas de Evaluación (Test): {len(X_test)}")

Reseñas de Entrenamiento: 132952
Reseñas de Evaluación (Test): 33239


In [ ]:
from sklearn.pipeline import Pipeline

# 1. Crear el pipeline que une la vectorización y el modelo
# Limitamos a max_features=10000 para no saturar la memoria RAM y quedarnos con el vocabulario más relevante
baseline_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

# 2. Entrenar el pipeline con el conjunto de entrenamiento
baseline_pipeline.fit(X_train, y_train)
print("¡Entrenamiento completado!")

¡Entrenamiento completado!


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Generar predicciones sobre el conjunto de evaluación (X_test)
y_pred = baseline_pipeline.predict(X_test)

# 2. Imprimir el reporte detallado de clasificación
print("--- REPORTE DE EVALUACIÓN (BASELINE ML) ---")
print(classification_report(y_test, y_pred, target_names=['Negativo (0)', 'Positivo (1)']))

# 3. Matriz de confusión
print("--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(y_test, y_pred))

--- REPORTE DE EVALUACIÓN (BASELINE ML) ---
              precision    recall  f1-score   support

Negativo (0)       0.24      0.81      0.37       817
Positivo (1)       0.99      0.94      0.96     32422

    accuracy                           0.93     33239
   macro avg       0.62      0.87      0.67     33239
weighted avg       0.98      0.93      0.95     33239

--- MATRIZ DE CONFUSIÓN ---
[[  662   155]
 [ 2054 30368]]


---

## Resumen de desiciones de la fase 2: Establecer un baseline

### 1. Definición del setup y estrategia de muestreo
* **Target Binario:** Se utilizó la columna `recommended` (`1` para Positivo, `0` para Negativo) como la variable objetivo ($y$) representando la intención explícita del usuario.
* **División del Dataset:** Se implementó un *split* **80% Entrenamiento / 20% Evaluación (Test)**.
* **Estratificación (`stratify=y`):** Debido al desbalance severo detectado en el EDA (97.5% positivo / 2.5% negativo), la estratificación garantizó que ambos subconjuntos conservaran exactamente la misma proporción de clases.
* **Criterio de Filtrado de Longitud:** Se decidió **no aplicar un filtro de corte de palabras** ($\le 3$ palabras) en esta fase. Si bien representan el 25% del dataset y aportan cierto ruido, términos directos como *"bad"*, *"good"* o *"masterpiece"* son señales de alta utilidad para modelos lineales y reflejan la distribución real que recibirá el sistema en producción.

### 2. Decisiones de Arquitectura del Pipeline (`Pipeline`)
* **Vectorización (TF-IDF):** Se seleccionó `TfidfVectorizer` sobre un *CountVectorizer* tradicional para penalizar términos de alta frecuencia sin valor predictivo (*"game"*, *"play"*) mediante el cálculo de la Frecuencia Inversa de Documentos (IDF).
* **Control de Vocabulario (`max_features=10000`):** Se fijó un límite de 10,000 términos para evitar la explosión de la matriz dispersa en memoria RAM producida por erratas y vocabulario marginal, reteniendo la señal semántica principal y reduciendo el tiempo de entrenamiento a escasos segundos.
* **Algoritmo Clasificador (`LogisticRegression`):** Se eligió una Regresión Logística por su estabilidad, rapidez e interpretabilidad con targets binarios.
* **Tratamiento del Desbalance (`class_weight='balanced'`):** En lugar de aplicar técnicas agresivas de *resampling* (como SMOTE o undersampling), se instruyó al modelo para penalizar matemáticamente con mayor severidad los errores cometidos sobre la clase minoritaria (Negativa).

---

### 3. Resultados del Modelo e Interpretación de Métricas

#### **Resultados Clave (Set de Evaluación - 33,239 muestras)**
* **Macro F1-Score (Métrica Principal del Proyecto):** `0.67`
* **Clase Negativa / Quejas (0):**
  * **Recall:** `0.81` (Se logró capturar el **81%** de las quejas reales: 662 de 817).
  * **Precision:** `0.24` (Solo el 24% de las predicciones de quejas fueron correctas, generando 2,054 falsas alarmas).
  * **F1-Score:** `0.37` (Piso de rendimiento de la clase minoritaria).
* **Accuracy Global:** `0.93` La metrica se puede considerar engañosa por el evidente sesgo del dataset hacia las reseñas positivas.

---

### 4. Conclusiones y Justificación para la Fase 3

1. **Cumplimiento del Benchmark:** Se estableció un "piso de rendimiento" ($F1\text{-Macro} = 0.67$) con un pipeline liviano, rápido y totalmente interpretable.
2. **Diagnóstico del Error:** La baja *Precision* en la clase negativa ($0.24$) se debe a que TF-IDF carece de comprensión del contexto oracional. Palabras como *"crash"* o *"bug"* gatillan predicciones negativas incluso cuando están usadas en contexto positivo o irónico (ej. *"Game had bugs at launch, but fixed now, 10/10"*).
3. **Pase a la Fase 3:** Para reducir la tasa de falsos positivos y categorizar aspectos simultáneos (rendimiento vs. contenido), queda formalmente justificado el salto a modelos de **Deep Learning / Transformers (DistilBERT)**.

---